In [3]:
import pandas as pd 
import torch 
import os 
import clearml 
import pathlib

In [4]:
BASE_PATH='/home/aanil/Data/aanil/side/yolo/datasets/Soccernet/tracking'

In [5]:
class PATHS: 
    train_path=pathlib.Path(f'{BASE_PATH}/train')
    test_path=pathlib.Path(f'{BASE_PATH}/test')


# Dataset 

In [44]:
import configparser
import copy
from pathlib import Path
from typing import List, Any
class PitchSenseDataset(torch.utils.data.Dataset):
    def __init__(self,roots:List[Any]):
        self.roots=roots
        self.imgs=[]
        for root in self.roots:
            root = Path(root)
            if not root.exists():
                continue
            for match_id in sorted(os.listdir(root)):
                match_path = root / match_id
                if not match_path.is_dir():
                    continue
                # print(match_path)
                sample = self._build_match(match_path)
                
                if sample:
                    self.imgs.extend(self._match_to_frames(sample))
    
    def _match_to_frames(self,sample):
        df_gt,df_det,path=sample['gt'],sample['det'],sample['match_path']
        frames=df_gt['frame'].unique()

        return [self._filter_dfs(sample,df_gt,df_det,path,frame) for frame in frames]

    def _filter_dfs(self,sample,df_gt,df_det,path,frame):
        filtered_gt=df_gt[df_gt['frame']==frame]
        filtered_det=df_det[df_det['frame']==frame]
        img_path=path / "img1" / f"{frame:06d}.jpg"
        
        return {
            "img_path": img_path,
            "gt": filtered_gt,
            "det": filtered_det,
            "config": sample['config']
        }

    def _build_match(self,match_path):
        tree=[f for f in match_path.iterdir()]
        game_config,seq_config = self.get_inis(tree)
        tree=[f for f in match_path.iterdir() if f.is_dir()]
        match_config=game_config | seq_config
        gt,det,tree=self.boiler_plate(tree,match_config)

        return {
            "match_path": match_path,
            "gt": gt,
            "det": det,
            "config": match_config
        }
    
    def _read_ini_to_dict(self,path: Path) -> dict:
        """Read an .ini file into a nested dictionary {section: {key: value}}"""
        parser = configparser.ConfigParser()
        parser.read(path)
        return {section: dict(parser[section]) for section in parser.sections()}

    def get_inis(self,tree):
        for sub_path in tree:
            if sub_path.is_file():
                if sub_path.stem == "gameinfo":
                    game_config = self._read_ini_to_dict(sub_path)['Sequence']
                elif sub_path.stem == "seqinfo":
                    seq_config = self._read_ini_to_dict(sub_path)['Sequence']
        return game_config,seq_config

    def get_mapper(self,match_config):
        tracklet_map = {}
        for key, value in match_config.items():  
            if key.startswith("trackletid_"):
                idx = int(key.split("_")[1])
                name = value.split(";")[0]  
                tracklet_map[idx] = name
        return tracklet_map

    def boiler_plate(self,tree,match_config):
        for item in copy.copy(tree):
            if item.stem=='gt':
                gt = pd.read_csv(item/"gt.txt", header=None)
                tree.remove(item)
            elif item.stem=='det':
                det = pd.read_csv(item/"det.txt", header=None)
                tree.remove(item)
        gt.columns = ['frame', 'track_id', 'x', 'y', 'w', 'h', 'class_id', 'f1','f2','f3']
        det.columns = ['frame', 'track_id', 'x', 'y', 'w', 'h', 'class_id', 'f1','f2','f3']
        tracklet_map=self.get_mapper(match_config)
        
        gt['name'] = gt['track_id'].map(tracklet_map)
        det['name'] = gt['track_id'].map(tracklet_map)
        return gt,det,tree


    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        return self.imgs[idx]

In [48]:
from torch.utils.data import random_split
# -------------------- Merge train + test -------------------- #
train_root = PATHS.train_path
test_root = PATHS.test_path

dataset = PitchSenseDataset([train_root, test_root])

# -------------------- Train/Val split -------------------- #
val_ratio = 0.15
val_size = int(len(dataset) * val_ratio)
train_size = len(dataset) - val_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))


print(f"Dataset samples: {len(dataset)}")
print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

Dataset samples: 79500
Train samples: 67575, Val samples: 11925


In [50]:
dataset[0]['gt']

,frame,track_id,x,y,w,h,class_id,f1,f2,f3,name
0,1,1,914,855,55,172,1,-1,-1,-1,player team left
578,1,2,917,575,32,122,1,-1,-1,-1,player team left
1328,1,3,956,557,53,133,1,-1,-1,-1,player team right
1688,1,4,1257,673,44,141,1,-1,-1,-1,player team right
2256,1,5,1888,400,30,101,1,-1,-1,-1,player team right
2738,1,6,1715,325,23,81,1,-1,-1,-1,player team right
3263,1,7,1520,478,50,82,1,-1,-1,-1,player team right
3950,1,8,1416,531,41,113,1,-1,-1,-1,player team right
4700,1,9,1091,378,33,86,1,-1,-1,-1,player team right
5382,1,10,978,333,22,75,1,-1,-1,-1,player team right
